# Ensemble Kalman inversion

Calibrate the two parameters of a decay model from noisy data, running the
same driver first in its sampling form and then in its optimization form.

## When to use this

Ensemble Kalman inversion (EKI) estimates the parameters of a forward model
using only evaluations of the model, never its derivatives, and evaluates
the whole ensemble at once. That suits expensive or black-box simulators.
Run to tempering level $\beta = 1$ it approximates the Bayesian posterior;
the approximation is exact for a linear model with a Gaussian prior and
biased otherwise (Iglesias, Law & Stuart, 2013; Ernst, Sprungk & Starkloff, 2015). Iterated with unit steps and
stopped by the discrepancy principle it becomes a derivative-free,
regularized optimizer (Iglesias, 2016).

## Setup

The forward model maps parameters $u = (a, r)$ to an exponential decay
observed at ten times $t_i$ equally spaced on $[0.1, 2]$:

$$
G(u)_i = a\, e^{-r t_i}, \qquad
y = G(u^\dagger) + e, \quad e \sim \mathcal N(0, R), \quad R = 0.05^2 I_{10},
$$

with $u^\dagger = (1.5, 1.2)$ and prior
$u \sim \mathcal N\big((1, 0.5),\ \operatorname{diag}(0.5, 0.5)\big)$.
EKI moves an ensemble through the tempered targets

$$
\pi_\beta(u) \propto \pi_0(u)\, e^{-\beta \Phi(u)}, \qquad
\Phi(u) = \tfrac12 \lVert R^{-1/2}(y - G(u)) \rVert^2 ,
$$

and a step from $\beta$ to $\beta + \delta$ is one ensemble Kalman update
with noise covariance $R/\delta$.

In [ ]:
import jax
import jax.numpy as jnp

import enskit
from enskit import kalman, toy
from enskit.algorithms import eki

problem = toy.exponential_decay()   # prior over "u", forward model, noise_cov, y

## Sampling form

The schedule picks each increment $\delta$ so that the importance weights
$w_j \propto e^{-\delta \Phi(u_j)}$ keep an effective sample size of half the
ensemble (Jasra et al., 2011), and stops at $\beta = 1$. The update rule is
Matheron's: each particle moves by $K(y - g_j - e_j)$ with a fresh noise draw
$e_j$ (Burgers, van Leeuwen & Evensen, 1998).

In [ ]:
state = eki.EKIState.from_prior(jax.random.key(0), problem.prior, n_particles=64)
result = eki.run(
    state, problem.forward, problem.y, problem.noise_cov,
    update_rule=kalman.Matheron(),
    schedule=eki.AdaptiveESSSchedule(ess_fraction=0.5),
)
print(result.status, result.n_evaluations, "evaluations")
print("posterior mean", result.ensemble.mean("u"), "truth", problem.u_true)

schedule_exhausted 6 evaluations
posterior mean [1.47542747 1.15540647] truth [1.5 1.2]


## Optimization form

Unit increments, repeated, drive the ensemble toward minimizers of
$\Phi$. The run stops by the discrepancy principle, at the first step where
$2\Phi(\bar g) \le \tau^2 N$, with $\bar g$ the mean prediction and $N = 10$
the data dimension (Iglesias, 2016). The deterministic square-root rule is
used here (Bishop, Etherton & Majumdar, 2001).

In [ ]:
fit = eki.run(
    state, problem.forward, problem.y, problem.noise_cov,
    update_rule=kalman.SymmetricSquareRoot(),
    schedule=eki.FixedSchedule.constant(1.0, n_steps=50),
    stop=eki.DiscrepancyStop(tau=1.0),
)
print(fit.status, "after", fit.n_evaluations, "evaluations:", fit.ensemble.mean("u"))

stopping_rule after 3 evaluations: [1.46251858 1.11982496]


## What to notice

The two forms differ only in the schedule and the stopping rule, not in the
driver. The sampling run spends its six evaluations reaching $\beta = 1$,
and its ensemble approximates the posterior. The optimization run stops as
soon as the fit is as good as the noise allows, with an ensemble mean close
to the parameters that generated the data.

In [ ]:
# checks
assert result.status == "schedule_exhausted"
assert abs(float(result.state.beta) - 1.0) < 1e-12
assert fit.status == "stopping_rule"
assert jnp.allclose(fit.ensemble.mean("u"), problem.u_true, atol=0.2)

## References

- Bishop, C. H., Etherton, B. J. & Majumdar, S. J. (2001). Adaptive sampling with the ensemble transform Kalman filter. Part I: Theoretical aspects. *Monthly Weather Review*, 129(3), 420–436.
- Burgers, G., van Leeuwen, P. J. & Evensen, G. (1998). Analysis scheme in the ensemble Kalman filter. *Monthly Weather Review*, 126(6), 1719–1724.
- Ernst, O. G., Sprungk, B. & Starkloff, H.-J. (2015). Analysis of the ensemble and polynomial chaos Kalman filters in Bayesian inverse problems. *SIAM/ASA Journal on Uncertainty Quantification*, 3(1), 823–851.
- Iglesias, M. A. (2016). A regularizing iterative ensemble Kalman method for PDE-constrained inverse problems. *Inverse Problems*, 32(2), 025002.
- Iglesias, M. A., Law, K. J. H. & Stuart, A. M. (2013). Ensemble Kalman methods for inverse problems. *Inverse Problems*, 29(4), 045001.
- Jasra, A., Stephens, D. A., Doucet, A. & Tsagaris, T. (2011). Inference for Lévy-driven stochastic volatility models via adaptive sequential Monte Carlo. *Scandinavian Journal of Statistics*, 38(1), 1–22.